# VMC2026 Track 2 — Baseline Pipeline (Kaggle)

QMOS (SpeechMOS) + EmoCat (emotion2vec) + **EMOS (emotion2vec target-prob, mặc định offline)** → gộp `answer.txt`.

**Trước khi chạy:** Accelerator = **GPU T4**, Internet = **On**.
- **+ Add Input** → tab Datasets → dataset Track 2 đã upload (Kaggle tự giải nén → có thư mục `vmc2026-track2/`).
- Với mặc định `EMOS_METHOD='emotion2vec'`: **KHÔNG cần** `GEMINI_API_KEY`. Chỉ cần Secrets khi đổi sang `'gemini'` (để có thêm VAD).

Chạy được ngay: **QMOS + EmoCat + EMOS** (chỉ cần wav + `metadata.csv` chứa cảm xúc target).

> ⚠️ Train phase: dự đoán tập **DEV** (`sets/dev.scp`, ~2730 mẫu). Thư mục `wav/` có cả train+dev nên KHÔNG glob hết — chỉ lấy đúng dev.scp.

## 0. Config — SỬA Ở ĐÂY

In [ ]:
import os, glob

# ── Data Track 2 trên Kaggle (dataset đã upload, KHÔNG có thư mục con lồng) ──
DATA_ROOT    = '/kaggle/input/vmc2026-track2-full'   # << slug dataset bạn upload
WAV_DIR      = f'{DATA_ROOT}/wav'
METADATA_CSV = f'{DATA_ROOT}/metadata.csv'      # wavID|emotion|transcript (KHÔNG header)
DEV_SCP      = f'{DATA_ROOT}/sets/dev.scp'      # danh sách wav tập DEV (tập cần nộp ở train phase)

# Test nhanh trên ESD: trỏ WAV_DIR vào ESD, đặt DEV_SCP=None và METADATA_CSV=None.
# WAV_DIR = '/kaggle/input/datasets/nguyenthanhlim/emotional-speech-dataset-esd/Emotion Speech Dataset'
# DEV_SCP = None; METADATA_CSV = None

LIMIT = 20   # << 20 = chạy THỬ nhanh. Đổi None để chạy TOÀN BỘ DEV rồi nộp.

# ── Cách tính EMOS ──────────────────────────────────────────────────────────
# 'emotion2vec': OFFLINE, MIỄN PHÍ (exp01, khuyến nghị) — P(cảm xúc target) từ emotion2vec → scale 1–5.
# 'gemini'     : LLM-as-judge qua Gemini API (cần GEMINI_API_KEY, tốn phí). Chỉ cách này có VAD.
EMOS_METHOD = 'emotion2vec'

OUT_DIR = '/kaggle/working'
RUN_QMOS, RUN_EMOCAT = True, True
_have_meta = bool(METADATA_CSV) and os.path.exists(METADATA_CSV)
RUN_EMOS = _have_meta                                # cả 2 cách đều cần target từ metadata
RUN_VAD  = _have_meta and EMOS_METHOD == 'gemini'    # VAD chỉ có ở Gemini
EMOTIONS5 = ['angry', 'happy', 'neutral', 'sad', 'surprised']

# Chuẩn hóa nhãn cảm xúc target (metadata) → đúng 1 trong 5 lớp của emotion2vec.
_EMO_ALIAS = {'angry':'angry','anger':'angry','happy':'happy','happiness':'happy','joy':'happy',
              'neutral':'neutral','calm':'neutral','sad':'sad','sadness':'sad',
              'surprise':'surprised','surprised':'surprised','surprising':'surprised'}
def norm_emotion(label):
    key = str(label).strip().lower()
    return _EMO_ALIAS.get(key, key if key in EMOTIONS5 else None)

def list_wavs(d):
    # Có DEV_SCP → đọc danh sách tên file tập DEV (wav nằm phẳng trong wav/).
    # Không có   → quét đệ quy mọi .wav (chế độ test ESD, lồng speaker/emotion).
    if DEV_SCP and os.path.exists(DEV_SCP):
        with open(DEV_SCP) as f:
            names = [ln.strip() for ln in f if ln.strip()]
        wavs = [os.path.join(d, n) for n in names]
    else:
        wavs = sorted(glob.glob(os.path.join(d, '**', '*.wav'), recursive=True))
    return wavs[:LIMIT] if LIMIT else wavs

print('WAV_DIR:', WAV_DIR, '| EMOS_METHOD:', EMOS_METHOD)
print('Số wav:', len(list_wavs(WAV_DIR)) if os.path.isdir(WAV_DIR) else '(chưa thấy thư mục)')
print('Chế độ DEV (dev.scp):', bool(DEV_SCP and os.path.exists(DEV_SCP)))
if METADATA_CSV and os.path.exists(METADATA_CSV) and DEV_SCP and os.path.exists(DEV_SCP):
    n_meta = sum(1 for _ in open(METADATA_CSV))
    n_dev  = sum(1 for _ in open(DEV_SCP))
    print(f'metadata.csv: {n_meta} dòng | dev.scp: {n_dev} dòng')

## 1. Cài đặt

In [ ]:
!pip install -q speechmos funasr librosa soundfile pandas google-genai loguru tqdm

## 2. QMOS — SpeechMOS (UTMOS, không cần fairseq)

In [ ]:
def run_qmos(wav_dir):
    import torch, librosa
    dev = 'cuda' if torch.cuda.is_available() else 'cpu'
    predictor = torch.hub.load('tarepan/SpeechMOS:v1.2.0', 'utmos22_strong', trust_repo=True).to(dev)  # << GPU
    print('QMOS device:', dev)
    scores, missing = {}, 0
    for w in list_wavs(wav_dir):              # w là đường dẫn đầy đủ
        if not os.path.exists(w):             # mẫu ESD/DailyTalk chưa lấy ngoài → bỏ qua, không crash
            missing += 1; continue
        wave, _ = librosa.load(w, sr=16000, mono=True)
        wave_t = torch.from_numpy(wave).unsqueeze(0).to(dev)   # << đưa input lên GPU
        scores[w] = float(predictor(wave_t, sr=16000).mean().item())
    if missing: print(f'[QMOS] Bỏ qua {missing} file thiếu (chưa có ESD/DailyTalk) → điểm mặc định.')
    return scores

qmos_scores = run_qmos(WAV_DIR) if RUN_QMOS else {}
print('QMOS xong:', len(qmos_scores))
list(qmos_scores.items())[:3]

## 3. EmoCat — emotion2vec+ large
Đã sửa bug bản gốc + lọc 5 lớp + chuẩn hóa tổng = 1.

In [ ]:
def run_emocat(wav_dir):
    import torch
    from funasr import AutoModel
    dev = 'cuda:0' if torch.cuda.is_available() else 'cpu'
    model = AutoModel(model='iic/emotion2vec_plus_large', hub='hf', device=dev)  # << chạy GPU
    print('EmoCat device:', dev)
    results, missing = {}, 0
    for w in list_wavs(wav_dir):              # w là đường dẫn đầy đủ
        if not os.path.exists(w):             # mẫu ESD/DailyTalk chưa lấy ngoài → bỏ qua
            missing += 1; continue
        rec = model.generate(w, granularity='utterance', extract_embedding=False)
        probs = {e: 0.0 for e in EMOTIONS5}
        for lab, sc in zip(rec[0]['labels'], rec[0]['scores']):
            name = lab.split('/')[-1]
            if name in probs:
                probs[name] = float(sc)
        total = sum(probs.values())
        if total > 0:
            probs = {k: v / total for k, v in probs.items()}
        results[w] = probs
    if missing: print(f'[EmoCat] Bỏ qua {missing} file thiếu (chưa có ESD/DailyTalk) → phân bố mặc định.')
    return results

emocat_probs = run_emocat(WAV_DIR) if RUN_EMOCAT else {}
print('EmoCat xong:', len(emocat_probs))
list(emocat_probs.items())[:2]

## 4. EMOS — emotion2vec target-prob (mặc định) hoặc Gemini
**emotion2vec (exp01, offline):** lấy P(cảm xúc target) từ emotion2vec (đã tính ở cell EmoCat), scale [0,1]→[1,5]. Chấm đủ 2.730 mẫu, KHÔNG cần API. SRCC chỉ quan tâm thứ hạng nên scale tuyến tính không đổi tương quan.

**Gemini (`EMOS_METHOD='gemini'`):** LLM-as-judge, cần `GEMINI_API_KEY` + credit; tự lọc metadata về DEV để đỡ tốn. Chỉ cách này có VAD.

In [ ]:
emos_scores, vad_scores = {}, {}   # key = TÊN FILE wav (uttID, có .wav)

# Đọc cảm xúc target từ metadata.csv → {stem: emotion_chuẩn}
def load_target_emotions():
    tgt = {}
    if not (METADATA_CSV and os.path.exists(METADATA_CSV)):
        return tgt
    with open(METADATA_CSV, encoding='utf-8') as f:
        for ln in f:
            parts = ln.strip().split('|')
            if len(parts) >= 2:
                stem = os.path.splitext(os.path.basename(parts[0]))[0]
                tgt[stem] = norm_emotion(parts[1])
    return tgt

target_map = load_target_emotions()
print('Nhãn cảm xúc target đọc được:', len(target_map))

if RUN_EMOS and EMOS_METHOD == 'emotion2vec':
    # ── EMOS OFFLINE: P(cảm xúc target) từ emotion2vec (cell EmoCat), scale [0,1]→[1,5] ──
    assert RUN_EMOCAT and emocat_probs, 'EMOS theo emotion2vec cần chạy cell EmoCat (mục 3) trước.'
    miss_t = miss_p = 0
    for w in list_wavs(WAV_DIR):
        name = os.path.basename(w)
        tgt = target_map.get(os.path.splitext(name)[0])
        probs = emocat_probs.get(w)
        if tgt is None:
            miss_t += 1; continue
        if not probs:
            miss_p += 1; continue
        emos_scores[name] = 1.0 + 4.0 * probs.get(tgt, 0.0)   # p=0→1 điểm, p=1→5 điểm
    if miss_t: print(f'[EMOS-e2v] {miss_t} mẫu thiếu nhãn target → mặc định 3.')
    if miss_p: print(f'[EMOS-e2v] {miss_p} mẫu thiếu prob emotion2vec → mặc định 3.')
    print(f'✅ EMOS (emotion2vec) cho {len(emos_scores)} mẫu — không cần API.')

elif RUN_EMOS or RUN_VAD:   # EMOS_METHOD == 'gemini'
    try:
        from kaggle_secrets import UserSecretsClient
        os.environ['GEMINI_API_KEY'] = UserSecretsClient().get_secret('GEMINI_API_KEY')
        print('Đã nạp GEMINI_API_KEY từ Secrets')
    except Exception as e:
        print('Chưa nạp được key:', e)

    # ── Lọc metadata.csv → CHỈ giữ mẫu thuộc DEV (tránh trả tiền Gemini cho mẫu train) ──
    dev_stems = {os.path.splitext(n.strip())[0] for n in open(DEV_SCP) if n.strip()}
    META_DEV = '/kaggle/working/metadata_dev.csv'
    kept = 0
    with open(METADATA_CSV) as fin, open(META_DEV, 'w') as fout:
        for line in fin:
            if not line.strip():
                continue
            stem = os.path.splitext(os.path.basename(line.split('|')[0].strip()))[0]
            if stem in dev_stems:
                fout.write(line); kept += 1
    print(f'metadata_dev.csv: {kept} dòng (kỳ vọng ~{len(dev_stems)})')

    GEMINI_ROWS = f'--end-row {LIMIT}' if LIMIT else ''
    !git clone -q https://github.com/voicemos-challenge/vmc2026-baselines.git /kaggle/working/vmc2026-baselines
    !cd /kaggle/working/vmc2026-baselines/track2/EMOS && python Gemini_EMOS.py --metadata-path $META_DEV --base-path $WAV_DIR --output-file /kaggle/working/emos.csv --workers 4 --resume $GEMINI_ROWS
    !cd /kaggle/working/vmc2026-baselines/track2/VAD && python Gemini_VAD.py --metadata-path $META_DEV --base-path $WAV_DIR --output-file /kaggle/working/vad.csv --workers 4 --resume $GEMINI_ROWS

    import pandas as pd
    if os.path.exists('/kaggle/working/emos.csv'):
        d = pd.read_csv('/kaggle/working/emos.csv'); emos_scores = dict(zip(d['uttID'], d['emos']))
    if os.path.exists('/kaggle/working/vad.csv'):
        d = pd.read_csv('/kaggle/working/vad.csv')   # cột chuẩn: uttID, val, aro, dom
        for _, r in d.iterrows():
            vad_scores[r['uttID']] = (r['val'], r['aro'], r['dom'])

    if emos_scores:
        dev_bases = {os.path.basename(w) for w in list_wavs(WAV_DIR)}
        if not (set(emos_scores) & dev_bases):
            print('⚠️ KEY LỆCH: uttID không khớp tên file dev → EMOS/VAD sẽ về mặc định!')
        else:
            print('✅ Key khớp — EMOS/VAD sẽ gộp đúng.')

print('EMOS:', len(emos_scores), '| VAD:', len(vad_scores))

## 5. Gộp answer.txt (tự bỏ cột thiếu)

In [ ]:
def fmt_cat(p):
    return '|'.join(f'{e}:{p[e]:.6g}' for e in EMOTIONS5)

def build_answer(out_path):
    wavs = list_wavs(WAV_DIR)
    have_cat = RUN_EMOCAT and len(emocat_probs) > 0
    have_vad = RUN_VAD and len(vad_scores) > 0
    cols = ['wav', 'QMOS', 'EMOS']
    if have_cat: cols.append('CAT')
    if have_vad: cols += ['VAL', 'ARO', 'DOM']
    with open(out_path, 'w') as f:
        f.write(','.join(cols) + '\n')
        for w in wavs:
            name = os.path.basename(w)            # tên file = cột wav & key của emos/vad
            row = [name, f"{qmos_scores.get(w, 3.0):.6g}", str(emos_scores.get(name, 3))]
            if have_cat: row.append(fmt_cat(emocat_probs.get(w, {e: 0.2 for e in EMOTIONS5})))
            if have_vad:
                v = vad_scores.get(name, (3, 3, 3)); row += [str(v[0]), str(v[1]), str(v[2])]
            f.write(','.join(row) + '\n')
    print(f'Ghi {len(wavs)} dòng → {out_path} | cột: {cols}')

answer_path = os.path.join(OUT_DIR, 'answer.txt')
build_answer(answer_path)
!head -3 {answer_path}

## 6. Validate + zip

In [ ]:
import csv
with open(answer_path) as f:
    rows = list(csv.reader(f))
header = rows[0]
assert header[0] == 'wav' and 'QMOS' in header and 'EMOS' in header, 'Header sai'
for i, r in enumerate(rows[1:], 2):
    assert len(r) == len(header), f'Dòng {i} sai số cột'
print(f'OK: {len(rows)-1} dòng, header = {header}')
!cd /kaggle/working && zip -j submission_track2.zip answer.txt && unzip -l submission_track2.zip